# Проверки данных

Факты о данных, на которых построены схема валидации и решение (`04_solution`). Каждая ячейка — проверка с `assert`:
если данные изменятся, ноутбук упадёт в месте, где перестало выполняться допущение.

In [1]:
import decimal
from pathlib import Path

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "dataset"
QUERY_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
              "search_infm_params_text", "search_category"]

train = pd.read_parquet(DATA_DIR / "train.parquet")
queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet")
print("train", train.shape, "| queries", queries.shape, "| items", items.shape)

train (497673, 19) | queries (2452, 6) | items (189212, 14)


## 1. Типы и ключи

In [2]:
assert train.shape[0] == 497_673 and queries.shape[0] == 2_452 and items.shape[0] == 189_212
for col in ["item_price", "item_latitude", "item_longitude"]:
    assert isinstance(train[col].dropna().iloc[0], decimal.Decimal), col
assert queries["query_id"].map(type).eq(str).all() and items["item_id"].map(type).eq(str).all()
assert queries["query_id"].is_unique and items["item_id"].is_unique
print("ID с ведущим нулём в корпусе:", items["item_id"].str.startswith("0").sum())

ID с ведущим нулём в корпусе: 11765


## 2. Дубликаты в train
Полные дубли (`search_*` + `item_id`) — повторные клики, удаляются перед валидацией.

In [3]:
in_dup_groups = train.duplicated(subset=QUERY_COLS + ["item_id"], keep=False)
full_dup = train.duplicated(subset=QUERY_COLS + ["item_id"])
print("строк в группах дублей:", in_dup_groups.sum(), "| удаляется (все, кроме первой в группе):", full_dup.sum())
assert in_dup_groups.sum() == 48_533 and full_dup.sum() == 30_619
train_dedup = train[~full_dup]
assert len(train_dedup) == 467_054

строк в группах дублей: 48533 | удаляется (все, кроме первой в группе): 30619


## 3. Что такое «запрос»
В train нет `query_id`: запрос — полная комбинация `search_*`. Релевантных объявлений на запрос — длинный хвост.

In [4]:
n_rel = train_dedup.groupby(QUERY_COLS, dropna=False)["item_id"].nunique()
print("уникальных текстов:", train_dedup["search_query"].nunique(), "| комбинаций:", len(n_rel))
print(n_rel.describe()[["mean", "50%", "max"]])
print("доля запросов с >50 релевантными (Recall@50<1 недостижим):", (n_rel > 50).mean())
assert n_rel.median() == 1

уникальных текстов: 74529 | комбинаций: 354463
mean      1.317638
50%       1.000000
max     278.000000
Name: item_id, dtype: float64
доля запросов с >50 релевантными (Recall@50<1 недостижим): 0.00011848909477152763


## 4. Пересечение train и бенчмарка
Объявления почти не пересекаются, а тексты запросов — заметно. Решение должно переносить знание по контенту.

In [5]:
common = set(train["item_id"]) & set(items["item_id"])
print("общих item_id:", len(common), f"({len(common) / len(items):.1%} корпуса)")
assert len(common) == 18_142

seen_text = queries["search_query"].isin(set(train["search_query"]))
full_seen = pd.MultiIndex.from_frame(queries[QUERY_COLS].astype(str)).isin(
    pd.MultiIndex.from_frame(train[QUERY_COLS].astype(str)))
print(f"тексты бенчмарка, встречающиеся в train: {seen_text.mean():.1%}; полная комбинация: {full_seen.mean():.1%}")
assert queries["search_query"].is_unique, "в бенчмарке все тексты уникальны -> валидируем 1 инстанс на текст"
assert abs(seen_text.mean() - 0.37) < 0.01

общих item_id: 18142 (9.6% корпуса)


тексты бенчмарка, встречающиеся в train: 37.0%; полная комбинация: 4.4%


## 5. Локация
Совпадение локации — сильный, но не абсолютный сигнал: мягкий буст, не фильтр. У части запросов бенчмарка в корпусе нет объявлений их локации — нужно гео-расстояние.

In [6]:
loc_match = (train_dedup["search_location_id"] == train_dedup["item_location_id"]).mean()
print(f"search_location_id == item_location_id в релевантных парах: {loc_match:.1%}")
assert 0.8 < loc_match < 0.85

per_loc = items["item_location_id"].value_counts()
n_in_loc = per_loc.reindex(queries["search_location_id"]).fillna(0)
print(f"запросов бенчмарка без объявлений своей локации в корпусе: {(n_in_loc == 0).mean():.1%}")
print("объявлений в локации запроса: медиана", n_in_loc.median())

search_location_id == item_location_id в релевантных парах: 82.2%
запросов бенчмарка без объявлений своей локации в корпусе: 17.4%
объявлений в локации запроса: медиана 1638.0


## 6. Категория
`search_category` практически константа и совпадает с категорией объявления — сигнала для ранжирования нет.

In [7]:
print(train["search_category"].value_counts().head(3).to_dict(), queries["search_category"].value_counts().to_dict())
same_cat = (train["search_category"] == train["item_category_id"]).mean()
print(f"search_category == item_category_id: {same_cat:.4%}")
assert same_cat > 0.999
print("микрокатегорий в train:", train["item_microcat_id"].nunique())

{114: 497635, 0: 34, 81: 2} {114: 2230, 0: 222}
search_category == item_category_id: 99.9894%
микрокатегорий в train: 212


## 7. Повторяющиеся заголовки
Много объявлений с одинаковым заголовком в разных городах. Отсюда одинаковые скоры BM25 у топа (это не баг) и заниженная офлайн-метрика: текстово идеальный кандидат из другого города не засчитывается.

In [8]:
titles = items["item_title_raw"].str.lower()
print(f"объявлений бенчмарка с неуникальным заголовком: {titles.duplicated().mean():.1%}")
print(titles[titles.str.contains("скупка телевизоров")].value_counts().head(3))

объявлений бенчмарка с неуникальным заголовком: 37.6%
item_title_raw
скупка телевизоров                          2
ремонт и скупка телевизоров в кингисеппе    1
скупка телевизоров целых и неисправных      1
Name: count, dtype: int64


## 8. Сплит для валидации

В `04_solution` train делится по полной комбинации полей `search_*`: один и тот же запрос не попадает в обе
части, но тексты могут повторяться — как у 37% запросов бенчмарка, чьи тексты есть в train.
Корпус для поиска — все объявления train, поэтому все правильные ответы валидации в нём есть.

In [9]:
train_dedup = train_dedup.reset_index(drop=True)
keys = train_dedup[QUERY_COLS].fillna("").astype(str).agg("\x1f".join, axis=1)
tr_idx, va_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=42).split(train_dedup, groups=keys))
assert not set(keys.iloc[tr_idx]) & set(keys.iloc[va_idx])

val = train_dedup.iloc[va_idx]
val_texts = val["search_query"].drop_duplicates()
share = val_texts.isin(set(train_dedup.iloc[tr_idx]["search_query"])).mean()
assert set(val["item_id"]) <= set(train_dedup["item_id"])
print(f"валидация: {val['search_query'].nunique()} текстов, знакомых по train {share:.1%} (в метрике взвешивается до 37%)")

валидация: 15839 текстов, знакомых по train 69.7% (в метрике взвешивается до 37%)
